bringing in cell hour dataset, aggregating to cell day, then redoing cell level proximity features

In [19]:
import h3
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from econml.dml import CausalForestDML
from xgboost import XGBRegressor, XGBClassifier

import json
import folium
from folium.features import GeoJsonTooltip


In [3]:
#bring in data
PATH = "cell_hour_for_meta_learners.parquet"  # adjust

df = pd.read_parquet(PATH)

df.size
df.columns

Index(['h3_cell', 'trips_start', 'total_trips', 'frac_exposed', 'n_stations',
       'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain',
       'cloud_cover', 'wind_speed_10m', 'strike_severity_daily_frac', 'hour',
       'is_am_peak', 'is_pm_peak', 'is_school_holiday', 'is_bank_holiday',
       'dist_nearest_tube_km', 'days_since_last_strike', 'cycle_infra_score',
       'days_to_next_strike', 'doy', 'is_weekend', 'y_log1p',
       'frac_near_capacity', 'y_per_station_log1p'],
      dtype='str')

In [6]:
df['day'] = pd.to_datetime(df["trips_start"]).dt.date

# Aggregate cell-hour to cell-day
cell_day = (
    df.groupby(["h3_cell", "day"])
    .agg(
        total_trips          = ("total_trips", "sum"),
        n_stations           = ("n_stations", "first"),
        frac_exposed         = ("frac_exposed", "max"),  # any exposure = treated
        temperature_2m       = ("temperature_2m", "mean"),
        relative_humidity_2m = ("relative_humidity_2m", "mean"),
        precipitation        = ("precipitation", "mean"),
        rain                 = ("rain", "mean"),
        doy                  = ("doy", "first"),
     #   dow                 = ("dow", "first"),
        cloud_cover          = ("cloud_cover", "mean"),
        wind_speed_10m      = ("wind_speed_10m", "mean"),
        strike_severity_daily_frac = ("strike_severity_daily_frac", "mean"),
        is_weekend           = ("is_weekend", "first"),
        is_bank_holiday      = ("is_bank_holiday", "first"),
        is_school_holiday    = ("is_school_holiday", "first"),
        dist_nearest_tube_km = ("dist_nearest_tube_km", "first"),
        days_to_next_strike  = ("days_to_next_strike", "first"),
        days_since_last_strike = ("days_since_last_strike", "first"),
        cycle_infra_score    = ("cycle_infra_score", "first")
    )
    .reset_index()
)

cell_day["y_per_station_log1p"] = np.log1p(
    cell_day["total_trips"] / cell_day["n_stations"]
)

print(f"Cell-day treatment rate: {(cell_day['frac_exposed'] > 0).mean():.4%}")
print(f"Treated cell-days      : {(cell_day['frac_exposed'] > 0).sum():,}")

Cell-day treatment rate: 0.7095%
Treated cell-days      : 1,008


In [8]:
# TfL publishes tube station coordinates. This is the 2024 static list.
# You can also fetch from: https://api.tfl.gov.uk/StopPoint/Mode/tube
_TUBE_STATIONS = pd.DataFrame([
    {"name": "Aldgate",            "lat": 51.5143, "lon": -0.0755},
    {"name": "Angel",              "lat": 51.5322, "lon": -0.1058},
    {"name": "Baker Street",       "lat": 51.5226, "lon": -0.1571},
    {"name": "Bank",               "lat": 51.5133, "lon": -0.0886},
    {"name": "Barbican",           "lat": 51.5204, "lon": -0.0979},
    {"name": "Bethnal Green",      "lat": 51.5272, "lon": -0.0549},
    {"name": "Bond Street",        "lat": 51.5142, "lon": -0.1494},
    {"name": "Cannon Street",      "lat": 51.5113, "lon": -0.0904},
    {"name": "Canary Wharf",       "lat": 51.5051, "lon": -0.0209},
    {"name": "Chancery Lane",      "lat": 51.5185, "lon": -0.1111},
    {"name": "Charing Cross",      "lat": 51.5080, "lon": -0.1247},
    {"name": "City Thameslink",    "lat": 51.5145, "lon": -0.1037},
    {"name": "Elephant & Castle",  "lat": 51.4943, "lon": -0.1001},
    {"name": "Embankment",         "lat": 51.5074, "lon": -0.1223},
    {"name": "Euston",             "lat": 51.5282, "lon": -0.1337},
    {"name": "Euston Square",      "lat": 51.5261, "lon": -0.1350},
    {"name": "Farringdon",         "lat": 51.5203, "lon": -0.1050},
    {"name": "Goodge Street",      "lat": 51.5199, "lon": -0.1344},
    {"name": "Holborn",            "lat": 51.5174, "lon": -0.1199},
    {"name": "Hyde Park Corner",   "lat": 51.5027, "lon": -0.1527},
    {"name": "Kings Cross",        "lat": 51.5308, "lon": -0.1238},
    {"name": "Knightsbridge",      "lat": 51.5014, "lon": -0.1607},
    {"name": "Lambeth North",      "lat": 51.4989, "lon": -0.1116},
    {"name": "Leicester Square",   "lat": 51.5113, "lon": -0.1281},
    {"name": "Liverpool Street",   "lat": 51.5178, "lon": -0.0823},
    {"name": "London Bridge",      "lat": 51.5055, "lon": -0.0861},
    {"name": "Marble Arch",        "lat": 51.5136, "lon": -0.1586},
    {"name": "Moorgate",           "lat": 51.5186, "lon": -0.0886},
    {"name": "Monument",           "lat": 51.5107, "lon": -0.0863},
    {"name": "Old Street",         "lat": 51.5263, "lon": -0.0873},
    {"name": "Oxford Circus",      "lat": 51.5154, "lon": -0.1417},
    {"name": "Paddington",         "lat": 51.5154, "lon": -0.1755},
    {"name": "Pimlico",            "lat": 51.4893, "lon": -0.1334},
    {"name": "Queensway",          "lat": 51.5108, "lon": -0.1873},
    {"name": "Russell Square",     "lat": 51.5232, "lon": -0.1244},
    {"name": "St James Park",      "lat": 51.4994, "lon": -0.1335},
    {"name": "St Pauls",           "lat": 51.5146, "lon": -0.0973},
    {"name": "Shoreditch High St", "lat": 51.5238, "lon": -0.0755},
    {"name": "Sloane Square",      "lat": 51.4924, "lon": -0.1565},
    {"name": "South Kensington",   "lat": 51.4941, "lon": -0.1738},
    {"name": "Southwark",          "lat": 51.5041, "lon": -0.1052},
    {"name": "Temple",             "lat": 51.5111, "lon": -0.1141},
    {"name": "Tottenham Court Rd", "lat": 51.5165, "lon": -0.1308},
    {"name": "Tower Hill",         "lat": 51.5098, "lon": -0.0766},
    {"name": "Vauxhall",           "lat": 51.4861, "lon": -0.1245},
    {"name": "Victoria",           "lat": 51.4965, "lon": -0.1447},
    {"name": "Warren Street",      "lat": 51.5243, "lon": -0.1388},
    {"name": "Waterloo",           "lat": 51.5036, "lon": -0.1143},
    {"name": "Westminster",        "lat": 51.5010, "lon": -0.1254},
])

In [9]:


def compute_cell_tube_features(
    cell_ids:      list,
    tube_stations: pd.DataFrame,   # your _TUBE_STATIONS DataFrame with lat/lon
    radii_km:      list = [0.5, 1.0, 2.0],
) -> pd.DataFrame:
    """
    For each H3 cell, compute tube proximity features using the cell centroid.

    This is the correct aggregation because it treats the cell as a geographic
    unit in its own right, rather than averaging over its constituent bike stations.

    Parameters
    ----------
    cell_ids      : list of H3 cell identifiers
    tube_stations : DataFrame with columns lat, lon
    radii_km      : radii at which to count nearby tube stations

    Returns
    -------
    DataFrame with one row per cell and columns:
        h3_cell, dist_nearest_tube_km, n_tube_within_500m,
        n_tube_within_1km, n_tube_within_2km
    """
    # ── Get centroid coordinates for each cell ────────────────────────────────
    cell_centroids = pd.DataFrame([
        {
            "h3_cell": cid,
            "lat":     h3.cell_to_latlng(cid)[0],
            "lon":     h3.cell_to_latlng(cid)[1],
        }
        for cid in cell_ids
    ])

    # ── Build KD-tree over tube station locations ─────────────────────────────
    # Convert to radians for approximate haversine via Euclidean in radian space
    tube_coords  = np.radians(tube_stations[["lat", "lon"]].values)
    tree         = cKDTree(tube_coords)

    cell_coords  = np.radians(cell_centroids[["lat", "lon"]].values)

    # ── Distance to nearest tube station ─────────────────────────────────────
    nearest_dist_rad, _ = tree.query(cell_coords, k=1)
    # Convert radians back to km
    cell_centroids["dist_nearest_tube_km"] = nearest_dist_rad * 6371.0

    # ── Count tube stations within each radius ────────────────────────────────
    for radius_km in radii_km:
        r_rad    = radius_km / 6371.0
        col_name = f"n_tube_within_{int(radius_km * 1000)}m"
        counts   = tree.query_ball_point(cell_coords, r=r_rad, return_length=True)
        cell_centroids[col_name] = counts

    return cell_centroids.drop(columns=["lat", "lon"])


# ── Apply to your cell-day panel ──────────────────────────────────────────────
unique_cells = cell_day["h3_cell"].unique().tolist()

cell_tube_features = compute_cell_tube_features(
    cell_ids      = unique_cells,
    tube_stations = _TUBE_STATIONS,
    radii_km      = [0.5, 1.0, 2.0],
)

print(cell_tube_features.describe().round(3))

# ── Merge back onto cell_day, replacing the old aggregated values ─────────────
# Drop the old station-aggregated versions first
cols_to_replace = [
    "dist_nearest_tube_km",
    "n_tube_within_500m",
    "n_tube_within_1km",
]
cell_day = cell_day.drop(
    columns=[c for c in cols_to_replace if c in cell_day.columns]
)
cell_day = cell_day.merge(cell_tube_features, on="h3_cell", how="left")

# Verify no NaNs introduced
print("\nNaN check after merge:")
print(cell_day[["dist_nearest_tube_km", "n_tube_within_500m"]].isna().sum())

       dist_nearest_tube_km  n_tube_within_500m  n_tube_within_1000m  \
count               132.000             132.000              132.000   
mean                  1.120               0.273                1.114   
std                   0.725               0.540                1.517   
min                   0.063               0.000                0.000   
25%                   0.526               0.000                0.000   
50%                   0.932               0.000                1.000   
75%                   1.649               0.000                1.250   
max                   3.574               3.000                6.000   

       n_tube_within_2000m  
count              132.000  
mean                 4.333  
std                  4.345  
min                  0.000  
25%                  1.000  
50%                  3.000  
75%                  7.000  
max                 16.000  

NaN check after merge:
dist_nearest_tube_km    0
n_tube_within_500m      0
dtype: int64


In [10]:
COL_TIME = "day"
COL_Y    = "total_trips"
COL_T    = "frac_exposed"

cell_day[COL_TIME] = pd.to_datetime(cell_day[COL_TIME])
cell_day[COL_Y] = pd.to_numeric(cell_day[COL_Y], errors="coerce")
cell_day[COL_T] = pd.to_numeric(cell_day[COL_T], errors="coerce").fillna(0).astype(int)

cell_day = cell_day.dropna(subset=[COL_TIME, COL_Y])
cell_day["y_log1p"] = np.log1p(cell_day[COL_Y].astype(float))

cell_day[[COL_TIME, COL_Y, COL_T, "y_log1p"]].head()

,day,total_trips,frac_exposed,y_log1p
0,2016-01-10,42,0,3.761200
1,2016-01-11,46,0,3.850148
2,2016-01-12,39,0,3.688879
3,2016-01-13,58,0,4.077537
4,2016-01-14,59,0,4.094345


In [11]:
SPLIT_DATE = "2018-01-01"

train = cell_day[cell_day[COL_TIME] < SPLIT_DATE].copy()
test  = cell_day[cell_day[COL_TIME] >= SPLIT_DATE].copy()

print("train:", train.shape, "test:", test.shape)

Y_train = train["y_log1p"].values
T_train = train[COL_T].values.astype(int)

Y_test = test["y_log1p"].values
T_test = test[COL_T].values.astype(int)

train: (93909, 25) test: (48162, 25)


In [12]:
train.columns

Index(['h3_cell', 'day', 'total_trips', 'n_stations', 'frac_exposed',
       'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain',
       'doy', 'cloud_cover', 'wind_speed_10m', 'strike_severity_daily_frac',
       'is_weekend', 'is_bank_holiday', 'is_school_holiday',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score',
       'y_per_station_log1p', 'dist_nearest_tube_km', 'n_tube_within_500m',
       'n_tube_within_1000m', 'n_tube_within_2000m', 'y_log1p'],
      dtype='str')

In [15]:
# Candidate numeric features (only keep those that exist)
num_cols = [
    "month",
    "year",
    "doy",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "is_weekend",
    "is_bank_holiday",
    'is_school_holiday',
    'dist_nearest_tube_km',
    'n_tube_within_500m', 
    'n_tube_within_1000m', 
    'strike_severity_daily_frac',
    'days_to_next_strike',
    'days_since_last_strike',
    'cycle_infra_score',
    'n_stations'
]


#cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in num_cols if c in train.columns]

#print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

pre = ColumnTransformer(
    transformers=[
      #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

Xtr = pre.fit_transform(train[num_cols])
Xte = pre.transform(test[num_cols])

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

num_cols: ['doy', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'is_weekend', 'is_bank_holiday', 'is_school_holiday', 'dist_nearest_tube_km', 'n_tube_within_500m', 'n_tube_within_1000m', 'strike_severity_daily_frac', 'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score', 'n_stations']
Xtr: (93909, 18) Xte: (48162, 18)


In [20]:
# Assuming your cell-hour dataframe is called cell_hour
# and has columns: frac_exposed (or strike_exposed), y_log1p, n_stations, h3_cell, trips_start

# 1. Basic treated vs control comparison
treated_mean = cell_day.loc[cell_day["frac_exposed"] > 0, "y_log1p"].mean()
control_mean = cell_day.loc[cell_day["frac_exposed"] == 0, "y_log1p"].mean()
naive_diff   = treated_mean - control_mean

print(f"Mean Y (treated cells) : {treated_mean:.4f}")
print(f"Mean Y (control cells) : {control_mean:.4f}")
print(f"Naive difference       : {naive_diff:.4f}")

# 2. Check n_stations distribution across treated vs control
print("\nStations per cell (treated vs control):")
print(cell_day.groupby("frac_exposed")["n_stations"].describe())

# 3. Check how many cells are ever treated
ever_treated = cell_day.groupby("h3_cell")["frac_exposed"].max()
print(f"\nCells ever treated  : {(ever_treated > 0).sum()}")
print(f"Cells never treated : {(ever_treated == 0).sum()}")
print(f"Fraction treated    : {(ever_treated > 0).mean():.3f}")

# 4. Check the outcome distribution
print("\nY distribution:")
print(cell_day.groupby("frac_exposed")["y_log1p"].describe())

# 5. Time of day — are you including night hours in treatment?
print("\nTreatment rate by hour:")# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

# ── Nuisance model for Y (outcome) ───────────────────────────────────────────
# Standard regression — no imbalance issue here.
# n_estimators deliberately modest; the causal forest adds further complexity.
xgb_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Nuisance model for T (propensity) ────────────────────────────────────────
# Binary target (0/1) fit as regression — outputs E[T|X] = P(T=1|X).
# scale_pos_weight is the key addition vs. Random Forest.
xgb_t = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── CausalForestDML ───────────────────────────────────────────────────────────
cf_xgb = CausalForestDML(
    model_y          = xgb_y,
    model_t          = xgb_t,
    n_estimators     = 200,       # more trees → smoother CATE surface
    min_samples_leaf = 50,
    max_features     = "auto",
    n_jobs           = -1,
    random_state     = 0,
)

cf_xgb.fit(Y_train, T_train, X=Xtr)
tau_cf_xgb   = cf_xgb.effect(Xte)
lb_xgb, ub_xgb = cf_xgb.effect_interval(Xte, alpha=0.05)

print(f"\nATE (XGB nuisance): {np.mean(tau_cf_xgb):.6f}")
print(f"95% CI:            [{np.mean(lb_xgb):.6f}, {np.mean(ub_xgb):.6f}]")
#print(cell_day.groupby("day")["frac_exposed"].mean().round(4))

Mean Y (treated cells) : 5.0076
Mean Y (control cells) : 4.6945
Naive difference       : 0.3131

Stations per cell (treated vs control):
                 count      mean       std  min  25%  50%  75%   max
frac_exposed                                                        
0             141063.0  1.584696  1.024718  1.0  1.0  1.0  2.0  10.0
1               1008.0  1.603175  1.017850  1.0  1.0  1.0  2.0   9.0

Cells ever treated  : 128
Cells never treated : 4
Fraction treated    : 0.970

Y distribution:
                 count      mean       std       min       25%       50%  \
frac_exposed                                                               
0             141063.0  4.694493  1.152716  0.693147  4.043051  4.859812   
1               1008.0  5.007628  1.146819  0.693147  4.391344  5.105945   

                   75%       max  
frac_exposed                      
0             5.509388  7.391415  
1             5.927592  7.380256  

Treatment rate by hour:
Treated: 607  |  Cont

In [22]:
test['treated'] = (test["frac_exposed"] > 0).astype(int)

In [23]:

# ── Step 1: Attach CATEs to your test set ─────────────────────────────────────
# tau_cf is your array of CATE estimates from CausalForestDML
# It has one value per row in your test set

test_with_cate = test.copy()
test_with_cate["cate_log"]    = tau_cf_xgb
test_with_cate["cate_pct"]    = np.expm1(tau_cf_xgb) * 100
test_with_cate["cate_lb_pct"] = np.expm1(lb_xgb) * 100
test_with_cate["cate_ub_pct"] = np.expm1(ub_xgb) * 100

# ── Step 2: Aggregate to cell level ──────────────────────────────────────────
# Each cell appears many times in the test set (once per day)
# Aggregate to a single CATE per cell for mapping

cell_cates = (
    test_with_cate
    .groupby("h3_cell")
    .agg(
        mean_cate_pct    = ("cate_pct",    "mean"),
        median_cate_pct  = ("cate_pct",    "median"),
        mean_lb_pct      = ("cate_lb_pct", "mean"),
        mean_ub_pct      = ("cate_ub_pct", "mean"),
        n_obs            = ("cate_pct",    "count"),
        n_treated_days   = ("treated",     "sum"),
        dist_nearest_tube = ("dist_nearest_tube_km", "mean"),
        n_tube_500m      = ("n_tube_within_500m",   "mean"),
    )
    .reset_index()
)

print(f"Cells with CATE estimates: {len(cell_cates)}")
print(f"\nCATE distribution across cells (%):")
print(cell_cates["mean_cate_pct"].describe().round(2))

# ── Step 3: Convert H3 cells to GeoJSON polygons ──────────────────────────────
def h3_cell_to_geojson_feature(row):
    """Convert a single H3 cell to a GeoJSON Feature with properties."""
    # h3.cell_to_boundary returns list of (lat, lon) tuples
    boundary = h3.cell_to_boundary(row["h3_cell"])
    # GeoJSON requires (lon, lat) ordering
    coordinates = [[lon, lat] for lat, lon in boundary]
    coordinates.append(coordinates[0])  # close the polygon

    return {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [coordinates],
        },
        "properties": {
            "h3_cell":          row["h3_cell"],
            "mean_cate_pct":    round(row["mean_cate_pct"], 2),
            "median_cate_pct":  round(row["median_cate_pct"], 2),
            "n_treated_days":   int(row["n_treated_days"]),
            "dist_nearest_tube": round(row["dist_nearest_tube"], 3),
            "n_tube_500m":      int(row["n_tube_500m"]),
            "ci_lower":         round(row["mean_lb_pct"], 2),
            "ci_upper":         round(row["mean_ub_pct"], 2),
        }
    }

geojson = {
    "type": "FeatureCollection",
    "features": [h3_cell_to_geojson_feature(row)
                 for _, row in cell_cates.iterrows()]
}

# ── Step 4: Build the Folium choropleth map ───────────────────────────────────
# Centre on central London
m = folium.Map(
    location=[51.507, -0.127],
    zoom_start=12,
    tiles="CartoDB positron"   # clean light basemap
)

# Colour scale — diverging around zero
# Red = negative CATE (bikes decrease), Blue = positive CATE (bikes increase)
cate_values  = cell_cates["mean_cate_pct"]
vmin         = cate_values.quantile(0.05)   # clip outliers
vmax         = cate_values.quantile(0.95)

colormap = folium.LinearColormap(
    colors=["#d73027", "#fee08b", "#ffffff", "#91bfdb", "#4575b4"],
    vmin=vmin,
    vmax=vmax,
    caption="Estimated CATE: % change in bike trips per station during strikes"
)

folium.GeoJson(
    geojson,
    style_function=lambda feature: {
        "fillColor":   colormap(feature["properties"]["mean_cate_pct"]),
        "color":       "grey",
        "weight":      0.5,
        "fillOpacity": 0.75,
    },
    tooltip=GeoJsonTooltip(
        fields=[
            "mean_cate_pct",
            "ci_lower",
            "ci_upper",
            "n_treated_days",
            "dist_nearest_tube",
            "n_tube_500m",
        ],
        aliases=[
            "Mean CATE (%)",
            "95% CI Lower (%)",
            "95% CI Upper (%)",
            "Strike days observed",
            "Dist to nearest tube (km)",
            "Tube stations within 500m",
        ],
        localize=True,
    ),
    name="CATE by H3 cell"
).add_to(m)

colormap.add_to(m)

# ── Step 5: Add tube station markers for context ──────────────────────────────
# So readers can see the relationship between CATE and tube proximity
for _, ts in _TUBE_STATIONS.iterrows():
    folium.CircleMarker(
        location=[ts["lat"], ts["lon"]],
        radius=4,
        color="#2c2c2c",
        fill=True,
        fill_color="#2c2c2c",
        fill_opacity=0.7,
        tooltip=ts.get("name", "Tube station"),
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save("strike_cate_map.html")
print("Map saved to strike_cate_map.html")

# ── Step 6: Supporting analysis for the article ───────────────────────────────
# Does CATE vary with tube proximity? This is the key heterogeneity story.

print("\nMean CATE by number of tube stations within 500m:")
print(
    cell_cates
    .groupby(pd.cut(cell_cates["n_tube_500m"],
                    bins=[-1, 0, 1, 2, 5, 100],
                    labels=["0", "1", "2", "3-5", "5+"]))
    ["mean_cate_pct"]
    .agg(["mean", "count"])
    .round(2)
)

print("\nMean CATE by distance to nearest tube station:")
print(
    cell_cates
    .groupby(pd.cut(cell_cates["dist_nearest_tube"],
                    bins=[0, 0.25, 0.5, 1.0, 2.0],
                    labels=["<250m", "250-500m", "500m-1km", "1-2km"]))
    ["mean_cate_pct"]
    .agg(["mean", "count"])
    .round(2)
)

Cells with CATE estimates: 132

CATE distribution across cells (%):
count    132.00
mean      -1.62
std        2.87
min       -9.12
25%       -3.16
50%       -2.03
75%       -0.48
max       10.52
Name: mean_cate_pct, dtype: float64
Map saved to strike_cate_map.html

Mean CATE by number of tube stations within 500m:
             mean  count
n_tube_500m             
0           -2.23    101
1            0.12     27
2            1.62      3
3-5          2.93      1

Mean CATE by distance to nearest tube station:
                   mean  count
dist_nearest_tube             
<250m              1.89      9
250-500m          -0.27     22
500m-1km          -1.85     38
1-2km             -2.45     45


In [25]:
# These are robust to the ATE instability because they describe
# relative patterns, not absolute magnitudes

# 1. Does CATE vary monotonically with tube proximity?
print("Mean CATE by n_tube_within_500m:")
print(
    test_with_cate
    .groupby("n_tube_within_500m")["cate_pct"]
    .agg(["mean", "count", "std"])
    .round(2)
)

# 2. Does the effect concentrate in peak hours?
# (if you still have hour in your cell-day data)
# print("\nMean CATE by is_am_peak:")
# print(test_with_cate.groupby("is_am_peak")["cate_pct"].mean().round(2))

# 3. What fraction of cells have positive estimated CATEs?
pct_positive = (test_with_cate["cate_pct"] > 0).mean()
print(f"\nFraction of cell-days with positive CATE: {pct_positive:.1%}")

# 4. Do cells near more tube stops have systematically higher CATEs?
# This is the key spatial heterogeneity story
correlation = test_with_cate["cate_pct"].corr(test_with_cate["n_tube_within_500m"])
print(f"\nCorrelation between CATE and n_tube_within_500m: {correlation:.3f}")

Mean CATE by n_tube_within_500m:
                    mean  count    std
n_tube_within_500m                    
0                  -2.23  36816  15.37
1                   0.12   9882  14.60
2                   1.62   1098  16.35
3                   2.93    366  18.28

Fraction of cell-days with positive CATE: 43.2%

Correlation between CATE and n_tube_within_500m: 0.074


In [38]:
# Approximate tube disruption score without line-level data
# Logic: on a day when 30% of London stations are exposed,
# a cell near 4 tube stops is more disrupted than one near 1

cell_day["tube_disruption_approx"] = (
    cell_day["strike_severity_daily_frac"] *   # how severe is today's strike network-wide
    cell_day["n_tube_within_500m"]             # how much tube access does this cell have
).clip(upper=1.0)

# On non-strike days this is 0.0 everywhere
# On strike days it scales with both city-wide severity and local tube density

In [39]:
cell_day.columns

Index(['h3_cell', 'day', 'total_trips', 'n_stations', 'frac_exposed',
       'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain',
       'doy', 'cloud_cover', 'wind_speed_10m', 'strike_severity_daily_frac',
       'is_weekend', 'is_bank_holiday', 'is_school_holiday',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score',
       'y_per_station_log1p', 'dist_nearest_tube_km', 'n_tube_within_500m',
       'n_tube_within_1000m', 'n_tube_within_2000m', 'y_log1p',
       'tube_disruption_approx', 'D_continuous', 'date_str', 'treated'],
      dtype='str')

In [46]:
# Ensure you have a proper date column at day level
cell_day["date_str"] = cell_day["day"].astype(str)
cell_day["treated"]  = (cell_day["frac_exposed"] > 0).astype(int)



# ── Define the control variables ──────────────────────────────────────────────
control_vars = [
    "y_per_station_log1p",
    "treated",
    "temperature_2m",
    "precipitation",
    "is_weekend",
    "is_bank_holiday",
    "is_school_holiday",
    "days_to_next_strike",
    "days_since_last_strike",
    "h3_cell",
    "date_str",
    'tube_disruption_approx',
    "n_tube_within_500m"
]

# Drop NaNs only from the columns the model will actually use
# This ensures the groups array and the model data have identical length
cell_day_clean = cell_day[control_vars].dropna().copy()

In [43]:
cell_day_clean.columns

Index(['y_per_station_log1p', 'treated', 'temperature_2m', 'precipitation',
       'is_weekend', 'is_bank_holiday', 'is_school_holiday',
       'days_to_next_strike', 'days_since_last_strike', 'h3_cell', 'date_str',
       'tube_disruption_approx'],
      dtype='str')

In [44]:
import statsmodels.formula.api as smf
import numpy as np

# Rebuild with continuous treatment
# tube_disruption_approx scales from 0 to n_tube_within_500m × severity
# Normalise to [0,1] for interpretability
cell_day_clean["D_continuous"] = (
    cell_day_clean["tube_disruption_approx"] /
    cell_day_clean["tube_disruption_approx"].max()
)

twfe_continuous = smf.ols(
    "y_per_station_log1p ~ D_continuous + C(h3_cell) + C(date_str)",
    data=cell_day_clean
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cell_day_clean["h3_cell"]}
)

ate_cont = np.expm1(twfe_continuous.params["D_continuous"]) * 100
ci_lo    = np.expm1(twfe_continuous.conf_int().loc["D_continuous", 0]) * 100
ci_hi    = np.expm1(twfe_continuous.conf_int().loc["D_continuous", 1]) * 100

print(f"Continuous treatment TWFE:")
print(f"  Effect of full local tube disruption : {ate_cont:+.2f}%")
print(f"  95% CI                               : [{ci_lo:.2f}%, {ci_hi:.2f}%]")
print(f"  p-value                              : {twfe_continuous.pvalues['D_continuous']:.4f}")

# Compare directly to your binary result
print(f"\nFor comparison:")
print(f"  Binary TWFE ATE : +10.27%  [4.62%, 16.22%]  p=0.0003")

Continuous treatment TWFE:
  Effect of full local tube disruption : +21.06%
  95% CI                               : [11.01%, 32.01%]
  p-value                              : 0.0000

For comparison:
  Binary TWFE ATE : +10.27%  [4.62%, 16.22%]  p=0.0003


In [47]:
# Better construction: requires knowing which lines were striking on each date
# If you have FOI data with line-level detail this is exact
# If not, use the following which is more interpretable than the current version

# Option A: Binary × tube density interaction (separates the two effects cleanly)
# This keeps the treatment binary but allows the effect to vary with tube proximity
# via an interaction term in the TWFE, rather than baking it into the treatment

twfe_interaction = smf.ols(
    """y_per_station_log1p ~ 
       treated                              
     + treated:n_tube_within_500m          
     + n_tube_within_500m                  
     + C(h3_cell) 
     + C(date_str)""",
    data=cell_day_clean
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cell_day_clean["h3_cell"]}
)

base_effect      = twfe_interaction.params["treated"]
interaction_coef = twfe_interaction.params["treated:n_tube_within_500m"]

print("Interaction model:")
print(f"  Base strike effect (0 tube stations within 500m):  "
      f"{np.expm1(base_effect)*100:+.2f}%")
print(f"  Additional effect per extra tube station within 500m: "
      f"{np.expm1(interaction_coef)*100:+.2f} pp")
print(f"  p-value (base):        {twfe_interaction.pvalues['treated']:.4f}")
print(f"  p-value (interaction): {twfe_interaction.pvalues['treated:n_tube_within_500m']:.4f}")

# Implied effects at different tube proximities
for n_tube in [0, 1, 2, 3]:
    effect = base_effect + interaction_coef * n_tube
    print(f"  Effect at n_tube={n_tube}: {np.expm1(effect)*100:+.2f}%")

Interaction model:
  Base strike effect (0 tube stations within 500m):  +5.17%
  Additional effect per extra tube station within 500m: +10.10 pp
  p-value (base):        0.0418
  p-value (interaction): 0.0000
  Effect at n_tube=0: +5.17%
  Effect at n_tube=1: +15.79%
  Effect at n_tube=2: +27.49%
  Effect at n_tube=3: +40.36%


In [48]:
#save cell day dataframe

cell_day.to_parquet("cell_day_with_tube_features.parquet", index=False)
